# ML-Adult-income

Réalisée par ZAHID Zainab

## Import des bibliothèques necessaires

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc, accuracy_score, classification_report, confusion_matrix

# Créer le dossier images s'il n'existe pas
os.makedirs('../images', exist_ok=True)
print('Bibliothèques chargées avec succès')

## Chargement du dataset

In [ ]:
columns = ['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital_status', 
           'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss',
           'hours_per_week', 'native_country', 'income']

df = pd.read_csv('../data/adult.data', names=columns, sep=', ', engine='python')
print(f'Dataset chargé: {df.shape}')
df.head()

## Exploration du dataset

In [ ]:
print('Informations du dataset:')
df.info()

In [ ]:
print('Valeurs manquantes:')
print(df.isnull().sum())

## Visualisation - Distribution des revenus

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='income')
plt.title('Distribution des revenus')
plt.tight_layout()
plt.savefig('../images/income_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

## Distribution de l'âge

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['age'], bins=30, kde=True)
plt.title('Distribution de l\'âge')
plt.tight_layout()
plt.savefig('../images/age_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

## Heures de travail par revenu

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='income', y='hours_per_week')
plt.title('Heures de travail par revenu')
plt.tight_layout()
plt.savefig('../images/hours_per_week.png', dpi=100, bbox_inches='tight')
plt.show()

## Prétraitement des données

In [ ]:
# Remplacement des ?
df.replace('?', np.nan, inplace=True)
print(f'Valeurs manquantes après remplacement:')
print(df.isnull().sum())

In [ ]:
# Suppression des valeurs nulles
df_clean = df.dropna()
print(f'Dataset avant nettoyage: {df.shape}')
print(f'Dataset après nettoyage: {df_clean.shape}')
df = df_clean

In [ ]:
# Sauvegarder les noms de colonnes avant encodage
feature_names = [col for col in df.columns if col != 'income']

# Encodage des colonnes catégorielles
encoder = LabelEncoder()
for column in df.columns:
    if df[column].dtype == 'object':
        df[column] = encoder.fit_transform(df[column].astype(str))

print('Encodage terminé')

In [ ]:
# Séparation de X et y
X = df.drop('income', axis=1)
y = df['income']

# Normalisation
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train set: {X_train.shape}, Test set: {X_test.shape}')

## Entraînement des modèles

In [ ]:
# KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)
acc_knn = accuracy_score(y_test, y_pred_knn)
print(f'Accuracy KNN: {acc_knn:.4f}')

In [ ]:
# Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
acc_dt = accuracy_score(y_test, y_pred_dt)
print(f'Accuracy Decision Tree: {acc_dt:.4f}')

In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)
print(f'Accuracy Logistic Regression: {acc_lr:.4f}')

In [ ]:
# Random Forest
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
print(f'Accuracy Random Forest: {acc_rf:.4f}')

## Comparaison des modèles

In [ ]:
results = pd.DataFrame({
    'Model': ['KNN', 'Decision Tree', 'Logistic Regression', 'Random Forest'],
    'Accuracy': [acc_knn, acc_dt, acc_lr, acc_rf]
})

print(results.to_string(index=False))

plt.figure(figsize=(10, 6))
sns.barplot(data=results, x='Model', y='Accuracy')
plt.title('Comparaison des modèles')
plt.ylabel('Accuracy')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../images/model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Matrice de confusion - Random Forest

In [ ]:
cm = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matrice de confusion - Random Forest')
plt.xlabel('Prédiction')
plt.ylabel('Réalité')
plt.tight_layout()
plt.savefig('../images/confusion_matrix_rf.png', dpi=100, bbox_inches='tight')
plt.show()

## Rapport détaillé

In [ ]:
print(classification_report(y_test, y_pred_rf))

## Importance des variables

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(feature_importance)

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance, x='Importance', y='Feature')
plt.title('Feature Importance - Random Forest')
plt.tight_layout()
plt.savefig('../images/feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

## Courbe ROC

In [ ]:
y_prob = rf.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.2f}')
plt.plot([0, 1], [0, 1], linestyle='--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Random Forest')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../images/roc_curve.png', dpi=100, bbox_inches='tight')
plt.show()

## Matrice de corrélation

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(df.corr(), cmap='coolwarm', center=0, annot=False)
plt.title('Matrice de corrélation')
plt.tight_layout()
plt.savefig('../images/correlation_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

## Validation croisée

In [ ]:
scores = cross_val_score(rf, X, y, cv=5)
print(f'CV Scores: {scores}')
print(f'Mean Score: {scores.mean():.4f}')
print(f'Std Dev: {scores.std():.4f}')